In [15]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Define the quantum device
n_qubits = 2  # Number of qubits
dev = qml.device("default.qubit", wires=n_qubits)

# Define the Quantum Circuit
@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))  # Encode inputs
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))  # Trainable quantum layer
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]  # Quantum measurement

# Corrected weight_shapes dictionary (only define trainable parameters)
weight_shapes = {"weights": (3, n_qubits, 3)}  # Shape of StronglyEntanglingLayers

# Hybrid Quantum-Classical Neural Network
class HybridQNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.quantum_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)  # Quantum Layer
        self.fc = nn.Linear(n_qubits, 1)  # Classical Linear Layer

    def forward(self, x):
        x = self.quantum_layer(x)  # Quantum layer processing
        x = self.fc(x)  # Classical layer processing
        return torch.sigmoid(x)  # Sigmoid activation for binary output

# Initialize the model
model = HybridQNN()
model

HybridQNN(
  (quantum_layer): <Quantum Torch Layer: func=quantum_circuit>
  (fc): Linear(in_features=2, out_features=1, bias=True)
)

In [16]:
# Example dataset (XOR problem)
X = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
y = torch.tensor([[0], [1], [1], [0]], dtype=torch.float32)  # XOR labels

# Training setup
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.1)

# Training loop
epochs = 500
for epoch in range(epochs):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# Test Prediction
print(np.round(model(X).detach().numpy(), 2))  # Output predictions

Epoch 0, Loss: 0.8353
Epoch 50, Loss: 0.2764
Epoch 100, Loss: 0.1318
Epoch 150, Loss: 0.0790
Epoch 200, Loss: 0.0537
Epoch 250, Loss: 0.0393
Epoch 300, Loss: 0.0303
Epoch 350, Loss: 0.0242
Epoch 400, Loss: 0.0199
Epoch 450, Loss: 0.0166
[[0.01]
 [0.99]
 [0.99]
 [0.01]]
